# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_minilink.ipynb)

**Write the equations once. Simulate, analyze, control, plan, optimize, learn.**

<table>
  <tr>
    <td width="25%" align="center"><img src="https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/cartpole_swingup.gif" width="100%"/></td>
    <td width="25%" align="center"><img src="https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/ur5_meshcat.gif" width="100%"/></td>
    <td width="25%" align="center"><img src="https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/racecar_mpc.gif" width="100%"/></td>
    <td width="25%" align="center"><img src="https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/rocket_landing.gif" width="100%"/></td>
  </tr>
  <tr>
    <td align="center">Cart-pole swing-up, trajectory optimization</td>
    <td align="center">UR5 arm, task-space impedance control</td>
    <td align="center">Racecar, model predictive control</td>
    <td align="center">Rocket landing, reinforcement learning</td>
  </tr>
</table>

Minilink is a Python framework for dynamical systems, control, optimization and learning. A
physical model is written once, as textbook equations, and becomes a `System`. Blocks wired
together are a `System` too. That one object is what every tool takes: classical and modern
control, optimal control (MPC, MPPI), reinforcement learning and parameter optimization.

This notebook runs the README section by section, with the animations the page cannot show,
then builds the four scenes above.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Block diagrams as code

A block is its equations:

$$\dot{x} = f(x, u, t; p), \qquad y = h(x, u, t; p)$$

Pure functions, arrays in and arrays out. Define `f` and `h`, and `@` closes the loop.

In [ ]:
import numpy as np

from minilink import PID, DynamicSystem


class SimplePendulum(DynamicSystem):
    def __init__(self):
        super().__init__(state_dim=2, input_dim=1, output_dim=1)

    def f(self, x, u, t=0, params=None):
        theta, dtheta = x
        return np.array([dtheta, u[0] - 9.81 * np.sin(theta)])

    def h(self, x, u, t=0, params=None):
        return x[:1]  # the sensor reads the angle


pid = PID(Kp=10.0, Ki=1.0, Kd=2.0)
diagram = pid @ SimplePendulum()
diagram.plot_diagram()

The loop is a system like any other. Released at one radian, the PID brings the pendulum back
to rest:

In [ ]:
diagram.of("plant").x0 = [1.0, 0.0]
diagram.compute_trajectory(tf=10.0)

In [ ]:
diagram.plot_trajectory()

## 2. Simulation, design and analysis from the same code

Example with a pendulum: its frequency response, the root locus of
$u = -k\,(\theta - \pi)$ at the upright, and a simulation. Each tool either works directly on
the nonlinear `f` or linearizes it around a nominal point.

In [ ]:
from minilink import Pendulum

plant = Pendulum()
up = np.array([np.pi, 0.0])

plant.plot_bode()

In [ ]:
plant.plot_root_locus(x_bar=up)

In [ ]:
plant.compute_trajectory(tf=6.0, x0=[2.0, 0.0])
plant.plot_trajectory()

In [ ]:
plant.animate()

Here an LQR designs a controller that balances it at the upright, and its Lyapunov function
bounds a region of attraction,
$V(x) = (x - \bar{x})^\top P (x - \bar{x}) \le c$.

In [ ]:
from minilink import lqr
from minilink.analysis import plot_region_of_attraction, region_of_attraction

balance = lqr(plant, Q=np.eye(2), R=np.eye(1), x_bar=up) @ plant
region = region_of_attraction(balance, x_bar=up)
plot_region_of_attraction(balance, region)

A start inside the region, a radian short of the upright, is brought back to balance:

In [ ]:
balance.compute_trajectory(tf=4.0, x0=[np.pi - 1.0, 0.0])
balance.animate()

## 3. A unified planning pipeline, from MPC to RL

All planning tools (LQR, MPC, VI, RRT, RL) work from a unified task description: the plant,
constraint sets and a cost function over the horizon $t_f$,

$$J = \int_0^{t_f} g(x, u, t)\, dt + h\big(x(t_f), t_f\big)$$

$$\dot{x} = f(x, u, t), \qquad x(0) = x_0, \qquad x(t) \in \mathcal{X}, \qquad u(t) \in \mathcal{U}, \qquad x(t_f) \in \mathcal{X}_f$$

The motor is too weak to lift the pendulum directly: it has to swing.

In [ ]:
from minilink import BallSet, PlanningProblem, QuadraticCost

cost = QuadraticCost.from_system(plant, x_bar=up, S=100 * np.eye(2))
swing_up = PlanningProblem(
    plant,
    x0=[0.0, 0.0],
    Xf=BallSet(up, 0.2),
    horizon=6.0,
    cost=cost,
    X=plant.state_box,
    U=plant.input_box,
)

Every planner returns a `PlanningSolution` whose `.controller` is a `System` block ready to
drop into a block diagram. **Trajectory optimization**
transcribes it into a nonlinear program; the live plot draws each iterate.

In [ ]:
from minilink import TrajectoryOptimizationPlanner

opt = TrajectoryOptimizationPlanner(swing_up, n_steps=60).solve(live_plot=True)

In [ ]:
opt.animate()

**Value iteration** solves the Bellman equation on a grid, sweep by sweep, and returns the
cost-to-go from every state.

In [ ]:
from minilink import DynamicProgrammingPlanner

vi_planner = DynamicProgrammingPlanner(
    swing_up, x_grid=(101, 101), u_grid=(11,), dt=0.05
)
vi = vi_planner.solve(record_history=True)
vi.plot_cost_to_go()

In [ ]:
vi_planner.animate_cost_to_go()

In [ ]:
vi.animate()

**RRT** grows a tree of motions until a branch reaches the goal.

In [ ]:
from minilink import RRTPlanner

rrt_planner = RRTPlanner(swing_up, seed=0)
rrt = rrt_planner.solve()
rrt_planner.plot_tree()

In [ ]:
rrt.animate()

**Reinforcement learning (PPO)** learns from the cost alone, from random starts. It is given the
same pendulum and a cost shaped for learning, $1 + \cos\theta$, lowest at the upright from every
angle; its answer is a neural network that is, again, a controller block.

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction, ReinforcementLearningPlanner, Uniform


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        return 1.0 + jnp.cos(x[0]) + 0.01 * x[1] ** 2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


random_starts = PlanningProblem(
    plant,
    cost=SwingUpCost(),
    horizon=np.inf,
    x0=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
    U=plant.input_box,
)
ppo_planner = ReinforcementLearningPlanner(
    random_starts,
    dt=0.05,
    features=lambda x: jnp.array([jnp.cos(x[0]), jnp.sin(x[0]), 0.1 * x[1]]),
    hidden=(32, 32),
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
)
ppo = ppo_planner.solve(n_timesteps=120_000)
ppo_planner.plot_learning_curve()

In [ ]:
learned = ppo.controller @ plant
learned.compute_trajectory(tf=8.0, dt=0.01, x0=[0.05, 0.0])
learned.animate()

**MPC** runs on a receding-horizon version of the problem: every 0.1 s it solves a 2 s
look-ahead again from the measured state. The animation draws the plan of each tick.

In [ ]:
from minilink.control import ModelPredictiveController
from minilink.planning.trajectory_optimization.mpc import mpc_animation_overlays

look_ahead = swing_up.replace(horizon=2.0, Xf=None)
mpc_planner = TrajectoryOptimizationPlanner(look_ahead, n_steps=20, backend="jax")
mpc = ModelPredictiveController(mpc_planner) % 0.1

closed_loop = mpc @ plant
run = closed_loop.compute_trajectory(tf=6.0)

In [ ]:
closed_loop.animate(run, overlays=mpc_animation_overlays(run, mpc, loop=closed_loop))

The task description can be stochastic, with parameters drawn from a distribution. `compare`
evaluates multiple controllers on the same task, the same 20 draws of the mass for each; on
JAX the draws run as one batch:

$$\mathbb{E}_{p \sim \mathcal{P}}\big[J(p)\big]$$

In [ ]:
uncertain = swing_up.replace(distributions={"plant": {"m": Uniform(0.8, 1.2)}})
comparison = uncertain.compare(VI=vi, RRT=rrt, TrajOpt=opt, MPC=mpc, n_trials=20)
print(comparison)

In [ ]:
comparison.plot_simulated_trajectory()

The open-loop plans miss the goal once the mass changes; the feedback laws reach it.

## 4. Optimization with respect to any parameter

Evaluating a controller on a task defines a function $J(p)$ of the whole parameter
tree—plant, controller, and cost—that can be differentiated through the simulation. For
example, an impedance law bringing the pendulum to rest under an uncertain mass $m$:

$$\dot{x} = f(x, u, t; p), \qquad u = -K_p\, \theta - K_d\, \dot{\theta}, \qquad J(p) = \int_0^{t_f} \left(x^\top Q\, x + u^\top R\, u\right) dt$$

With `backend="jax"`, the evaluator gives the expected cost and its gradient with respect
to every parameter in the tree:

$$\mathbb{E}_m\big[J(p)\big], \qquad \nabla_p\, \mathbb{E}_m\big[J(p)\big]$$

In [ ]:
import jax

from minilink import Impedance

rest_cost = QuadraticCost.from_system(plant, R=0.1 * np.eye(1))
rest = PlanningProblem(plant, x0=[2.0, 0.0], horizon=5.0, cost=rest_cost)
task = rest.replace(distributions={"plant": {"m": Uniform(0.8, 1.2)}})
controller = Impedance(Kp=1.0, Kd=1.0)
evaluator = task.compile(controller, backend="jax")


def expected_cost(params):
    return evaluator.evaluate(params, n_trials=20).objective


gradient = jax.grad(expected_cost)(evaluator.params)
gradient["controller"]

The gains, named by their paths in the tree, become the decision variables
$z = (K_p, K_d)$ of a program; an `Optimizer` solves it:

$$\min_{z = (K_p,\, K_d)} \ \mathbb{E}_m\big[J(z; p)\big] \quad \text{s.t.} \quad z_{\text{lower}} \le z \le z_{\text{upper}}$$

In [ ]:
from minilink import Optimizer

gains = ["controller.Kp", "controller.Kd"]
bounds = {"controller.Kp": (0.0, 50.0), "controller.Kd": (0.0, 20.0)}
program = task.program(controller, gains, bounds=bounds, n_trials=20)
best = Optimizer(program).solve()
print(best)

This program is a `MathematicalProgram`, the one shape every transcription lowers to
(collocation, a tuning task, a policy search):

$$\min_{z} \ J(z; p) \quad \text{s.t.} \quad h(z; p) = 0, \quad g(z; p) \ge 0, \quad z_{\text{lower}} \le z \le z_{\text{upper}}$$

The tuned gains written back into the tree, and the loop they give:

In [ ]:
tuned = evaluator.unpack(best.z, gains)
tuned_run = evaluator.trajectory(tuned)
evaluator.loop.plot_trajectory(tuned_run)

In [ ]:
evaluator.loop.animate(tuned_run)

## 5. Compiled, differentiated, batched

With `backend="jax"`, catalog plants—and any custom `f` written with `array_module(x, u)`—
compile to pure JAX functions: `jax.jit`, `jax.grad`, and `jax.vmap` apply directly, and the
tools above use them under the hood. A thousand pendulums, released from a thousand angles,
in one call:

In [ ]:
starts = np.column_stack([np.linspace(-3.0, 3.0, 1000), np.zeros(1000)])
t, swings = plant.rollout_batch(starts, tf=5.0, dt=0.005)

Each row of `swings` is one run, sampled at its 1001 times; the angles, one faint line per
pendulum:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(t, swings[:, :, 0].T, color="C0", alpha=0.02)
plt.xlabel("t [s]")
plt.ylabel("θ [rad]")
plt.show()

## 6. The four scenes

The animations at the top are the same few verbs on larger systems.

### Cart-pole swing-up, trajectory optimization

The cart starts two metres left with the pole hanging, and ends still under the upright pole:
the goal is a single state, reached exactly.

In [ ]:
from minilink import CartPole, SingletonSet
from minilink.graphical.catalog import camera_matrix

cartpole = CartPole()
cartpole.inputs["u"].lower[0] = -10.0  # [N]
cartpole.inputs["u"].upper[0] = 10.0  # [N]
x_goal = np.array([0.0, np.pi, 0.0, 0.0])

cartpole_swing_up = PlanningProblem(
    cartpole,
    horizon=4.0,
    x0=[-2.0, 1.0, 0.0, 0.0],
    Xf=SingletonSet(x_goal),
    cost=QuadraticCost.from_system(
        cartpole, Q=np.diag([1.0, 1.0, 0.0, 0.0]), R=np.diag([0.01]), x_bar=x_goal
    ),
    U=cartpole.input_box,
)
cartpole_plan = TrajectoryOptimizationPlanner(
    cartpole_swing_up,
    n_steps=40,
    transcription="direct_collocation",
    backend="jax",
    method="slsqp",
).solve()
cartpole_plan.plot_trajectory()

In [ ]:
cartpole_plan.animate(camera=camera_matrix(target=(-1.0, 0.75, 0.0), scale=3.6))

### UR5 arm, task-space impedance control

A spring and a damper between the hand and a target that glides through four points,
$f = K_p (r - p) - K_d \dot{p}$, sent to the six joints as $\tau = J(q)^\top f + g(q)$. The spring
is soft, so the hand trails its target and the force arrow shows. The 3-D view turns with the
mouse.

In [ ]:
from minilink import TaskImpedance, TrajectorySource, UR5Manipulator

waypoints = np.array(
    [
        [-0.5, 0.0, 0.5],
        [0.0, 0.5, 0.5],
        [-0.3, 0.3, 0.2],
        [-0.5, -0.3, 0.4],
        [-0.5, 0.0, 0.5],
    ]
)  # [m]

# The target holds at each waypoint, then glides to the next
t_knots = [0.0]
p_knots = [waypoints[0]]
for p_from, p_to in zip(waypoints[:-1], waypoints[1:]):
    t_knots += [t_knots[-1] + 0.5, t_knots[-1] + 2.0]
    p_knots += [p_from, p_to]
target = TrajectorySource(t_knots, np.array(p_knots).T)

arm = UR5Manipulator()
q_start = arm.inverse_kinematics(waypoints[0], q_guess=[0.0, -1.0, 1.2, -1.4, 0.0, 0.0])
arm.x0 = arm.q2x(q_start, np.zeros(6))

impedance = TaskImpedance(
    arm,
    gravity_comp=True,
    Kp=[100.0, 100.0, 100.0],
    Kd=[20.0, 20.0, 20.0],
    show_task_force=True,
    task_force_scale=0.08,  # [m/N]
)
arm_loop = target >> impedance @ arm
arm_loop.plot_diagram()

In [ ]:
arm_run = arm_loop.compute_trajectory(tf=t_knots[-1] + 0.5, n_samples=260)
arm_loop.plot_trajectory(signals=((target, "y"), (arm, "p")))

In [ ]:
arm_loop.animate(arm_run, renderer="meshcat", is_3d=True)

### Racecar, model predictive control

The 1/10 UdeS racecar laps a circuit past two cones. The MPC plans on the car's drivetrain
bicycle model, believing in a little more grip than the floor gives; its cost holds a cruise
speed, the lane and a clearance from the cones, each a field turned into a cost.

In [ ]:
from minilink.catalog import UdeSRacecarDyn
from minilink.planning import (
    ReferenceTrack,
    Scene,
    Sphere,
    car_outline,
    circuit_waypoints,
    from_waypoints,
    mount,
    point_probe,
    quadratic_excess,
    quadratic_hinge,
)

# The circuit, its lane and two cones leaning into it

width = 8.0  # [m]

path = circuit_waypoints(length=12.0, width=width, radius=2.0)
track = ReferenceTrack(from_waypoints(path), half_width=0.6)
cones = [(2.0, 0.5 * width - 0.13), (-2.0, 0.5 * width + 0.13)]
scene = Scene(obstacles=[Sphere(c, 0.10 + 0.15) for c in cones])

In [ ]:
# The model the MPC plans on: the friction it believes, and the bounds of its wheel rate,
# steering and power states

v_cruise = 4.5  # [m/s]
p_cruise = 10.0  # [W]

model = UdeSRacecarDyn(named_ports=False)
model.params["terrain"]["friction"] = 0.7
r_wheel = model.params["r_r"]
model.state.lower[6:9] = [0.0, -model.params["delta_max"], -model.params["P_max"]]
model.state.upper[6:9] = [
    3.0 * v_cruise / r_wheel,
    model.params["delta_max"],
    model.params["P_max"],
]

# The cost: cruise, stay near the line and inside the lane, keep clear of the cones

centre = mount(point_probe(), on=model)
body = mount(car_outline(length=0.34, width=0.20, margin=0.02), on=model)
x_cruise = np.zeros(model.n)
x_cruise[[3, 6, 8]] = [v_cruise, v_cruise / r_wheel, p_cruise]
racing_cost = (
    QuadraticCost.from_system(
        model,
        Q=np.diag([0.0, 0.0, 0.0, 4.0, 1.0, 0.0, 0.0, 2.0, 0.0]),
        R=np.diag([0.0005, 60.0]),
        S=np.diag([0.0, 0.0, 0.0, 8.0, 2.0, 0.0, 0.0, 2.0, 0.0]),
        x_bar=x_cruise,
        u_bar=np.array([p_cruise, 0.0]),
    )
    + track.distance_field(centre).as_cost(
        weight=60.0, shaping=quadratic_excess(threshold=0.05)
    )
    + track.corridor_field(body).as_cost(
        weight=200.0, shaping=quadratic_hinge(threshold=0.0)
    )
    + scene.clearance_field(body).as_cost(
        weight=400.0, shaping=quadratic_hinge(threshold=0.30)
    )
)

In [ ]:
# The start: on the bottom straight, a little left of the line, at cruise

first = int(np.argmin(np.abs(path[:, 0]) + np.abs(path[:, 1] + 0.5 * width)))
along = path[first + 1] - path[first]
heading = np.arctan2(along[1], along[0])
start = path[first] + 0.15 * np.array([-np.sin(heading), np.cos(heading)])
x_start = np.array(
    [*start, heading, v_cruise, 0.0, 0.0, v_cruise / r_wheel, 0.0, p_cruise]
)

# The MPC: a 1.2 s look-ahead solved by collocation every 0.1 s, one tick late

racing = PlanningProblem(
    model,
    x0=x_start,
    cost=racing_cost,
    horizon=1.2,
    X=model.state_box,
    U=model.input_box,
)
racing_planner = TrajectoryOptimizationPlanner(
    racing,
    n_steps=12,
    transcription="direct_collocation",
    backend="jax",
    method="slsqp",
    options={"maxiter": 60, "ftol": 1.0},
)
racing_mpc = ModelPredictiveController(racing_planner, delay=True)

In [ ]:
from minilink.graphical.catalog import racecar_skin_3d
from minilink.planning.trajectory_optimization.mpc import plant_run

# The car: the same model on a floor with less grip, in its 3-D look

car = UdeSRacecarDyn(named_ports=False)
car.params["terrain"]["friction"] = 0.6
car.x0 = x_start.copy()
car.camera_scale = 2.1  # [m]
car.skin = racecar_skin_3d

racing_loop = (racing_mpc % 0.1) @ car
lap = racing_loop.compute_trajectory(tf=10.3, dt=0.002)
frames = lap.resample(n_samples=155)
racing_overlays = mpc_animation_overlays(
    lap,
    racing_mpc,
    loop=racing_loop,
    scene=scene,
    track=track,
    traj=plant_run(frames, racing_loop),
)

In [ ]:
racing_loop.animate(frames, renderer="meshcat", speed=2.0, overlays=racing_overlays)

### Rocket landing, reinforcement learning

PPO learns to land a rocket from about 20 m, from random starts, its thrust and gimbal bounded.
The failure cost charges a run that leaves the allowed states. Training is the longest cell of
the notebook, about a minute on a laptop.

In [ ]:
from minilink import Gaussian, Rocket

x_landed = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0])

rocket = Rocket()
rocket.params["inertia"] = 1000.0
weight = rocket.params["mass"] * rocket.params["gravity"]
rocket.inputs["u"].lower = np.array([0.0, -0.05])
rocket.inputs["u"].upper = np.array([2.0 * weight, 0.05])
rocket.state.lower = np.array([-100.0, 0.0, -1.0, -50.0, -50.0, -5.0])
rocket.state.upper = np.array([100.0, 200.0, 1.0, 50.0, 50.0, 5.0])


class LandingCost(CostFunction):
    Q = 0.1 * np.diag([1.0, 1.0, 10.0, 0.1, 0.1, 1.0])
    R = 0.1 * np.diag([1e-8, 1.0])

    def g(self, x, u, t=0.0, params=None):
        dx = x - x_landed
        return dx @ self.Q @ dx + u @ self.R @ u

    def h(self, x, t=0.0, params=None):
        return 0.0


landing = PlanningProblem(
    rocket,
    cost=LandingCost(),
    horizon=15.0,
    x0=Gaussian([0.0, 20.0, 0.0, 0.0, 0.0, 0.0], [10.0, 8.0, 0.3, 2.0, 2.0, 0.3]),
    penalize_failure=True,
    failure_cost=100.0,
    X=rocket.state_box,
    U=rocket.input_box,
)
pilot = ReinforcementLearningPlanner(
    landing,
    dt=0.05,
    features=lambda x: (x - x_landed) * np.array([0.25, 0.25, 1.0, 0.25, 0.25, 1.0]),
    hidden=(64, 64),
    method="ppo",
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=1e-3,
    gamma=0.995,
    log_std_init=-1.0,
    seed=0,
)
pilot.solve(n_timesteps=4_000_000)
pilot.plot_learning_curve()

In [ ]:
descent = pilot.controller @ rocket
descent.compute_trajectory(tf=7.0, dt=0.01, x0=[6.0, 14.0, 0.0, 0.0, 0.0, 0.0])
descent.animate(camera=camera_matrix(target=(3.0, 7.0, 0.0), scale=8.0))

## The Minilink backbone

Three objects hold it together: the model (`System`), the task (`PlanningProblem`) and the
solver (`MathematicalProgram` and `Optimizer`).

![The Minilink Backbone](https://raw.githubusercontent.com/alx87grd/minilink/main/docs/_static/pipeline_map.png)

## Where to go next

- [`basics/`](basics/01_minilink.ipynb): model, connect, simulate, design a controller.
- [`core/`](core/01_systems.ipynb): the objects behind the tools, one by one.
- [`advanced/`](advanced/01_discrete_and_hybrid.ipynb): sampled loops, optimal control, MPC, reinforcement learning, differentiable loops.

On your own machine, `plant.game()` puts the keyboard on a system's input in real time.